# Silver cleaning — ARQLMED (SCADA signals to per-transformer series)

This notebook turns the raw bronze SCADA signals into a clean, regular, per-transformer
time series with rolling features, saved as `hive_metastore.silver.silver_arqlmed`.

The main steps are:

1. **Load and basic cleanup** — parse the timestamp, drop unused columns, type the value.
2. **Filter to the signals of interest** — keep only the relevant `ID` patterns and valid
   readings.
3. **Put readings on a regular grid** — snap each reading to the nearest 15 minutes.
4. **Pivot** — turn the two per-transformer signals (suffix `U--` and `I--`) into two
   columns, `VOLTAGE` and `CURRENT`.
5. **Quality filter and fill** — keep transformers with enough complete history, then
   forward-fill small gaps.
6. **Save** to silver.

## 1. Load and basic cleanup

Load the common functions, then read the bronze table. The preview and schema confirm the source loaded correctly.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
df = spark.read.table("hive_metastore.bronze.bronze_arqlmed")
display(df.limit(5))
df.printSchema()

Parse the `DATA_` text field into a real timestamp (the helper adds a `DATE` column).

In [0]:
df = parse_ts(df, "DATA_")

display(df)

Drop columns not needed downstream, including the original `DATA_` now that `DATE` exists.

In [0]:
df = df.drop("CIM_UID", "OPR_STM_FONTE", "OPR_ID_LOAD", "OPR_TM_LOAD", "OPR_TIPO_OPERACAO", "POLO", "NTIME", "DATA_")

Type the `STATE` value as a number and round it to 2 decimals.

In [0]:
df = cast_manual(df, "STATE", 'double')


In [0]:
df = df.withColumn("STATE", round("STATE", 2))

Profile the cleaned data so far.

In [0]:
dbutils.data.summarize(df)

**Numbers from a previous run:**

| rows | min timestamp | max timestamp |
|---|---|---|
| ~897M | 2023-07-04T10:25:19Z | 2024-07-03T00:00:38Z |

## 2. Filter to the signals of interest

Keep only the `ID`s that match the wanted pattern and have valid readings.

> The pattern below (position 2 is `P` or `S`, position 7 is not `-`, `9`, or `4`) is a
> fixed rule with no stated reason in the code. If it appears in the methodology, the
> choice of positions and values may need a short justification.

In [0]:
df = df.filter(
    (col("ID").substr(2, 1).isin("P", "S")) &  # Check position 2
    (~col("ID").substr(7, 1).isin("-", "9", "4"))  # Check position 7
)

Narrow further to IDs ending in `U--` or `0II--` (the two signal suffixes used later).

In [0]:
# Generate df_arqlmed_U- using filter and like operations
filtered_df = df.filter((col("ID").rlike("U--$")) | (col("ID").rlike("0II--$")))


Drop negative readings (keep `STATE >= 0`).

In [0]:
filtered_df = filtered_df.filter(filtered_df["STATE"] >= 0)

Profile the filtered data.

In [0]:
dbutils.data.summarize(filtered_df)

about 383M rows remain after filtering.

> Exploration

In [0]:
# data inicio dados = 2023-11-07 data fim = 2023-12-03

# Specify the equipment ID and metric you want to plot
equipment_id = "QSVLN-5505-0TU--"
metric_column = "STATE"
start_date = "2023-07-04"  # Start date for filtering
end_date = "2024-07-03"    # End date for filtering (for a week-long span)

# Filter the DataFrame for the specified equipment ID and date range
graph_df = filtered_df.filter((filtered_df["ID"] == equipment_id) &
                                 (filtered_df["DATE"] >= start_date) &
                                 (filtered_df["DATE"] <= end_date))

# Sort the DataFrame by the timestamp
sorted_df = graph_df.orderBy("DATE")

# Convert PySpark DataFrame to Pandas DataFrame
pd_df = sorted_df.select("DATE", metric_column).toPandas()


# Plot the data using Plotly
fig = px.line(pd_df, x="DATE", y=metric_column, title=f"Metric {metric_column} for Equipment ID {equipment_id}")

# Add a horizontal line representing the limit
fig.add_shape(
    type="line",
    x0=pd_df["DATE"].min(),
    x1=pd_df["DATE"].max(),
    line=dict(color="Red", width=2, dash="dash"),
    name="High Limit"
)

# Update layout to include the limit in the legend
fig.update_layout(
    shapes=[dict(
        type="line",
        x0=pd_df["DATE"].min(),
        x1=pd_df["DATE"].max(),
        line=dict(color="Red", width=2, dash="dash")
    )],
    annotations=[dict(
        x=pd_df["DATE"].mean(),
        xref="x",
        yref="y",
        showarrow=False,
        font=dict(color="Red")
    )]
)

# Show the plot
fig.show()

## 3. Put readings on a regular 15-minute grid

SCADA readings do not land on exact clock intervals, so each timestamp is snapped to the
nearest 15 minutes. This gives every transformer a regular series, which the pivot and the
rolling windows rely on.

Keep only ID prefixes that have more than one distinct full `ID` (i.e. transformers that report more than one signal), so the pivot has something to combine.

In [0]:
df_a = filtered_df.withColumn("ID_prefix", substring(col("ID"), 1, 11))

In [0]:
# Group by the first 12 characters and filter groups with more than one distinct ID
temp_df = df_a.groupBy("ID_prefix").agg(count_distinct("ID").alias("distinct_count")) \
    .filter(col("distinct_count") > 1)

# Join back with the original DataFrame to filter the relevant rows
df_a = df_a.join(temp_df, "ID_prefix")

# Show the result
df_a.display()

Snap each timestamp to the nearest 15-minute mark into a new `DATE_15M` column, then preview the before/after.

In [0]:
interval_s = 15 * 60  # 900

df_a = df_a.withColumn(
    "DATE_15M",
    F.to_timestamp(
        F.from_unixtime(
            (F.round(F.unix_timestamp(col("DATE")) / interval_s) * interval_s).cast("long")
        )
    )
)

display(df_a.select("DATE", "DATE_15M").limit(20))


Replace `DATE` with the snapped `DATE_15M`.

In [0]:
df_a = (
    df_a
    .drop("DATE")
    .withColumnRenamed("DATE_15M", "DATE")
)

Check that every timestamp now sits exactly on a 15-minute boundary (all offsets should be 0).

In [0]:
display(
    df_a.select(
        ((F.unix_timestamp(col("DATE")) % 900)).alias("offset_s")
    )
    .groupBy("offset_s")
    .count()
    .orderBy("offset_s")
)


## 4. Pivot the two signals into columns

Each transformer reports two signals, distinguished by the `ID` suffix (`U--` and `I--`).
This step pivots them into two columns per transformer per timestamp.

In [0]:
p_df = df_a.withColumn("ID_prefix", substring(col("ID"), 1, 12)) \
           .withColumn("suffix", substring(col("ID"), -3, 3))

# List of columns to pivot
columns_to_pivot = ["STATE"]


pivoted_dfs = []
for column in columns_to_pivot:
    pivoted_df = p_df.groupBy("ID_prefix", "DATE").pivot("suffix").agg(F.first(column))
    
    # Check the column names in the pivoted DataFrame
    print(pivoted_df.columns)
    
    # Rename columns based on expected pivot values
    pivoted_df = pivoted_df.withColumnRenamed("U--", f"{column}_T").withColumnRenamed("I--", f"{column}_I")
    
    # Ensure that renaming reflects actual column names after pivot
    if 'STATE_T' in pivoted_df.columns and 'STATE_I' in pivoted_df.columns:
        pivoted_df = pivoted_df.withColumnRenamed("STATE_T", "VOLTAGE").withColumnRenamed("STATE_I", "CURRENT")
    
    pivoted_dfs.append(pivoted_df)

In [0]:
display(pivoted_df)

## 5. Quality filter and forward-fill

Drop transformers with too little history or too many gaps, then fill the small remaining
gaps.

In [0]:
null_rows = pivoted_df.filter(
    F.col("current").isNull() | F.col("voltage").isNull()
)

display(null_rows)

In [0]:
display(df.filter(col("ID") == "LSPOV-3318-0TU--"))

In [0]:
display(pivoted_df.filter(col("CURRENT") > 1000))

Count rows and null rate per transformer.

In [0]:
min_rows = 34000
max_null_rate = 0.01  # try 0.01 first, change to 0.03 if you will fill later

id_stats = (
    pivoted_df
    .groupBy("ID_prefix")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum((F.col("current").isNull() | F.col("voltage").isNull()).cast("int")).alias("n_null_rows")
    )
    .withColumn("null_rate", F.col("n_null_rows") / F.col("n_rows"))
    .orderBy(F.desc("n_null_rows"))
)

display(id_stats)

Keep only transformers that clear both thresholds.

In [0]:
good_ids = (
    id_stats
    .filter((F.col("n_rows") >= min_rows) & (F.col("null_rate") <= max_null_rate))
    .select("ID_prefix")
)

pivoted_df = pivoted_df.join(good_ids, on="ID_prefix", how="left_semi")

Re-check the per-transformer null stats after the filter.

In [0]:
id_stats = (
    pivoted_df
    .groupBy("ID_prefix")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum((F.col("current").isNull() | F.col("voltage").isNull()).cast("int")).alias("n_null_rows")
    )
    .withColumn("null_rate", F.col("n_null_rows") / F.col("n_rows"))
    .orderBy(F.desc("n_null_rows"))
)

display(id_stats)

Forward-fill `current` and `voltage`: carry the last known value forward over gaps. The
window runs from the start up to the current row only (`unboundedPreceding` to `0`), so
the fill uses past and present values, never future ones.

In [0]:
# set these to your real column names
id_col = "ID_prefix"
ts_col = "DATE"

w_ffill = (
    Window.partitionBy(id_col)
          .orderBy(F.col(ts_col))
          .rowsBetween(Window.unboundedPreceding, 0)
)

pivoted_df = (
    pivoted_df
    .withColumn("current", F.last("current", ignorenulls=True).over(w_ffill))
    .withColumn("voltage", F.last("voltage", ignorenulls=True).over(w_ffill))
)

display(pivoted_df.select(id_col, ts_col, "current", "voltage").orderBy(id_col, ts_col).limit(50))

Re-check null stats after filling.

In [0]:
id_stats = (
    pivoted_df
    .groupBy("ID_prefix")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum((F.col("current").isNull() | F.col("voltage").isNull()).cast("int")).alias("n_null_rows")
    )
    .withColumn("null_rate", F.col("n_null_rows") / F.col("n_rows"))
    .orderBy(F.desc("n_null_rows"))
)

display(id_stats)

> **Exploration — not part of the pipeline.** Two more outlier spot-checks (large
> current values). Diagnostic only.

In [0]:
display(pivoted_df.filter(col("CURRENT") > 1000))

In [0]:
display(pivoted_df.filter(col("ID_prefix") == "LPFANH5515-0").filter(col("current") > 300))

## 7. Save to silver

Save the feature-enriched series as Delta. `overwrite` plus `overwriteSchema` makes the
cell safely re-runnable.

**Target table:** `hive_metastore.silver.silver_arqlmed`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_arqlmed"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

Save the table.

In [0]:
(
    pivoted_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


Read the saved table back and profile it as a final check.

In [0]:
df = spark.read.table("hive_metastore.silver.silver_arqlmed")
display(df.limit(5))
df.printSchema()

In [0]:
dbutils.data.summarize(df)